# RT Notebook 15: Articulation and Continuation Geometry

Purpose: move from predictive articulation importance to bounded theorem-candidate analysis.

Primary outputs are implication tables, counterexample catalogs, and proof obligations, not classifier scores.


In [ ]:
from __future__ import annotations

import json, zipfile
from pathlib import Path
from typing import Callable, Dict, List

import pandas as pd

SPEC_ID = 'NB15_ARTICULATION_CONTINUATION_GEOMETRY_THEORY_001'
NB13_ZIP_CANDIDATES = [
    Path('/content/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
    Path('/content/drive/MyDrive/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
    Path('departments/colab/results/NB13_TOPOLOGICAL_INVARIANTS_CONTINUATION_GEOMETRY_002.zip'),
]
NB13_ZIP = next((p for p in NB13_ZIP_CANDIDATES if p.exists()), None)
if NB13_ZIP is None:
    raise FileNotFoundError('Notebook 13 v2 result zip not found.')

OUTPUT_DIR = Path('/content') / SPEC_ID if Path('/content').exists() else Path('departments/colab/results') / SPEC_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Using NB13 archive:', NB13_ZIP)
print('Output dir:', OUTPUT_DIR)


## Load Invariant Table


In [ ]:
with zipfile.ZipFile(NB13_ZIP) as zf:
    with zf.open('topological_invariants.parquet') as f:
        invariants = pd.read_parquet(f)
    nb13_manifest = json.loads(zf.read('manifest.json').decode('utf-8'))

required = ['config_id', 'geometry', 'J', 'articulation_point_count', 'articulation_min_depth', 'articulation_max_depth', 'terminal_basin_count', 'reachability_closure_count', 'closure_lattice_node_count']
missing = [c for c in required if c not in invariants.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}')

print('Rows:', len(invariants))
print(invariants['geometry'].value_counts(dropna=False))


## Theorem-Candidate Evaluation Helpers


In [ ]:
def evaluate_implication(name: str, antecedent, consequent, directionality: str, statement: str, proof_obligation: str) -> Dict[str, object]:
    ant = antecedent(invariants)
    con = consequent(invariants)
    subset = invariants[ant]
    counter = invariants[ant & ~con]
    if len(subset) == 0:
        status = 'INCONCLUSIVE_EMPTY_ANTECEDENT'
    elif len(counter) == 0:
        status = 'SUPPORTED_WITHIN_ENUMERATED_DOMAIN'
    else:
        status = 'REFUTED_BY_COUNTEREXAMPLE'
    return {
        'candidate_id': name,
        'statement': statement,
        'directionality': directionality,
        'antecedent_count': int(len(subset)),
        'counterexample_count': int(len(counter)),
        'status': status,
        'example_counterexample_config_ids': counter['config_id'].head(10).tolist(),
        'proof_obligation': proof_obligation,
        'claim_ceiling': 'C2_CANDIDATE_AFTER_GOVERNED_REVIEW'
    }

def geom(label: str):
    return lambda df: df['geometry'].eq(label)

def articulation_present(df):
    return df['articulation_point_count'] > 0

def articulation_absent(df):
    return df['articulation_point_count'] == 0

def articulation_depth_nonnegative(df):
    return df['articulation_min_depth'] >= 0

def multiple_terminal_basins(df):
    return df['terminal_basin_count'] > 1

def closure_lattice_nontrivial(df):
    return df['closure_lattice_node_count'] > 1


## Evaluate Candidate Statements


In [ ]:
candidates = [
    evaluate_implication(
        'ART_OBS_REQUIRES_ARTICULATION_001',
        geom('obstruction'), articulation_present, 'necessary_condition_candidate',
        'Every obstruction has at least one articulation point.',
        'Define obstruction and articulation on finite continuation graphs; prove or refute obstruction -> articulation_present.'
    ),
    evaluate_implication(
        'ART_UNIV_EXCLUDES_ARTICULATION_001',
        geom('universal_reconvergence'), articulation_absent, 'exclusion_candidate',
        'Universal reconvergence excludes articulation points.',
        'Check whether universal reconvergence can coexist with articulation in finite continuation graphs.'
    ),
    evaluate_implication(
        'ART_OBS_DEPTH_EXISTS_001',
        geom('obstruction'), articulation_depth_nonnegative, 'depth_condition_candidate',
        'Every obstruction has a defined minimum articulation depth.',
        'Define minimum articulation depth and test obstruction -> depth is defined.'
    ),
    evaluate_implication(
        'ART_OBS_BASIN_SPLIT_001',
        geom('obstruction'), multiple_terminal_basins, 'deeper_invariant_candidate',
        'Obstruction implies multiple terminal basins.',
        'Determine whether articulation is a consequence of basin decomposition rather than a primitive cause.'
    ),
    evaluate_implication(
        'ART_OBS_CLOSURE_NONTRIVIAL_001',
        geom('obstruction'), closure_lattice_nontrivial, 'deeper_invariant_candidate',
        'Obstruction implies a nontrivial closure lattice.',
        'Relate obstruction to future-closure lattice structure before assigning causal language.'
    )
]

candidate_table = pd.DataFrame(candidates)
candidate_table.to_csv(OUTPUT_DIR / 'articulation_theorem_candidates.csv', index=False)
candidate_table


## Counterexamples and Proof Obligations


In [ ]:
counterexamples = {
    row['candidate_id']: row['example_counterexample_config_ids']
    for _, row in candidate_table.iterrows()
    if row['counterexample_count'] > 0
}
(OUTPUT_DIR / 'articulation_counterexamples.json').write_text(json.dumps(counterexamples, indent=2), encoding='utf-8')

proof_obligations = candidate_table[[
    'candidate_id', 'statement', 'status', 'proof_obligation', 'claim_ceiling'
]].copy()
proof_obligations.to_csv(OUTPUT_DIR / 'proof_obligations.csv', index=False)

manifest = {
    'notebook': 'RT Notebook 15',
    'title': 'Articulation and Continuation Geometry',
    'spec_id': SPEC_ID,
    'source_archive': str(NB13_ZIP),
    'rows_loaded': int(len(invariants)),
    'candidate_count': int(len(candidate_table)),
    'claim_ceiling': 'C1 before governed output induction; C2 candidate only after result zip registration and review',
    'interpretation_constraint': 'Bounded theorem-candidate analysis only; no formal theorem status, causal claim, or external physical validation.'
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest
